In [17]:
pip install pytesseract

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
from PIL import Image
import pytesseract
import numpy as np
import os

In [ ]:
def preprocess_image(image_path):
    """
    Preprocess the image for better OCR accuracy
    """
    # Read the image
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding to preprocess the image
    gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    # Apply deskewing if needed
    gray = deskew(gray)
    
    return gray

In [ ]:
def deskew(image):
    """
    Deskew the image to improve OCR accuracy
    """
    coords = np.column_stack(np.where(image > 0))
    angle = cv2.minAreaRect(coords)[-1]
    
    # The ray angle in minAreaRect is normally [45, 135]
    # So we need to adjust for this
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    
    # Rotate the image to deskew
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    return rotated

In [ ]:
def extract_words_with_metadata(image_path, ground_truth_path=None):
    """
    Extract words from the image and save as individual cropped images
    
    Args:
    image_path (str): Path to the input image
    ground_truth_path (str, optional): Path to the ground truth text file
    
    Returns:
    list of dictionaries containing word extraction metadata
    """
    # Ensure output directory exists
    output_dir = 'extracted_words'
    os.makedirs(output_dir, exist_ok=True)
    
    # Preprocess the image
    preprocessed_image = preprocess_image(image_path)
    
    # Use Tesseract to get word bounding boxes
    custom_config = r'--oem 3 --psm 11 -c tessedit_create_box=1 -c tessedit_create_hocr=1'
    
    # Perform OCR with word-level detection
    details = pytesseract.image_to_data(preprocessed_image, 
                                        output_type=pytesseract.Output.DICT, 
                                        config=custom_config)
    
    # Load ground truth words if provided
    ground_truth_words = []
    if ground_truth_path:
        with open(ground_truth_path, 'r', encoding='utf-8') as f:
            ground_truth_words = [line.strip() for line in f]
    
    # Read the original image for cropping
    original_image = cv2.imread(image_path)
    
    # List to store word extraction metadata
    word_metadata = []
    
    # Process each detected word
    for i in range(len(details['text'])):
        # Filter out empty or insignificant detections
        if int(details['conf'][i]) > 30 and len(details['text'][i].strip()) > 0:
            # Extract word details
            word = details['text'][i]
            x = details['left'][i]
            y = details['top'][i]
            w = details['width'][i]
            h = details['height'][i]
            
            # Crop the word
            word_image = original_image[y:y+h, x:x+w]
            
            # Generate filename
            # Use ground truth word if available, otherwise use detected word
            filename = f"{word}_{i}.png"
            if ground_truth_words and i < len(ground_truth_words):
                filename = f"{ground_truth_words[i]}_{i}.png"
            
            # Save the cropped word image
            filepath = os.path.join(output_dir, filename)
            cv2.imwrite(filepath, word_image)
            
            # Store metadata
            word_metadata.append({
                'word': word,
                'x': x,
                'y': y,
                'width': w,
                'height': h,
                'confidence': details['conf'][i],
                'filepath': filepath
            })
    
    return word_metadata

In [21]:
def main():
    # Example usage
    image_path = r"C:\Users\prana\Downloads\WhatsApp Image 2025-03-26 at 23.58.11_e36f83b3.jpg"  # Replace with your image path
    ground_truth_path = r"C:\Users\prana\Downloads\Y pues en la celestial Jeru- salen.txt"  # Optional: path to ground truth text file
    
    # Extract words
    results = extract_words_with_metadata(image_path, ground_truth_path)
    
    # Print metadata for extracted words
    for word_info in results:
        print(f"Word: {word_info['word']}")
        print(f"Filepath: {word_info['filepath']}")
        print(f"Confidence: {word_info['confidence']}")
        print("---")

if __name__ == "__main__":
    main()

Word: Y
Filepath: extracted_words\Y_4.png
Confidence: 91
---
Word: pues
Filepath: extracted_words\pues_5.png
Confidence: 95
---
Word: en
Filepath: extracted_words\en_6.png
Confidence: 94
---
Word: la
Filepath: extracted_words\la_7.png
Confidence: 88
---
Word: celeftial
Filepath: extracted_words\celeftial_8.png
Confidence: 92
---
Word: Jeru-.
Filepath: extracted_words\Jeru-._9.png
Confidence: 54
---
Word: falén
Filepath: extracted_words\falén_13.png
Confidence: 51
---
Word: no
Filepath: extracted_words\no_14.png
Confidence: 95
---
Word: ha
Filepath: extracted_words\ha_15.png
Confidence: 93
---
Word: mudado
Filepath: extracted_words\mudado_16.png
Confidence: 91
---
Word: de
Filepath: extracted_words\de_17.png
Confidence: 93
---
Word: condicion
Filepath: extracted_words\condicion_18.png
Confidence: 90
---
Word: vueftra
Filepath: extracted_words\vueftra_22.png
Confidence: 57
---
Word: Benignidad
Filepath: extracted_words\Benignidad_23.png
Confidence: 86
---
Word: ,
Filepath: extracted_word

In [20]:
# import pytesseract
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'  # Update this path if necessary

In [18]:
# pytesseract.pytesseract.tesseract_cmd = r'C:\Users\prana\AppData\Local\Tesseract-OCR\tesseract.exe'